In [1]:
# 04_llm_explanations.ipynb
# Decision-level explanation layer (T-2, T-3). Non-circular, verifiable evaluation:
# the model's true drivers are NOT given to the LLM. For each adverse credit
# decision the LLM must (a) infer the denial reasons and (b) propose a concrete
# numeric recourse. We then score:
#   - faithfulness : overlap between the LLM's reasons and the model's actual
#                    local drivers (signed coef x standardized value), @k=3;
#   - recourse_valid: whether APPLYING the LLM's proposed change actually flips
#                    the model's decision to approve (true contestability);
#   - protected_mention: whether the explanation cites a protected attribute.
# Requires OPENAI_API_KEY in a .env at the repo ROOT; run LOCALLY. Single cell.

import os, json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from dotenv import load_dotenv, find_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
N_CASES = 15          # number of adverse decisions evaluated per model tier

def find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "notebooks").is_dir():
            return c
    return p.parent if p.name == "notebooks" else p
ROOT = find_root()
DATA, FIG, TAB = ROOT / "data", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85"})
def savefig(fig, stem):
    for e in ("png", "pdf"):
        fig.savefig(FIG / (stem + "." + e), dpi=600, bbox_inches="tight")
    plt.close(fig)
def savetable(df, stem, index=False):
    df.to_csv(TAB / (stem + ".csv"), index=index)
    try:
        df.to_latex(TAB / (stem + ".tex"), index=index, float_format="%.3f")
    except Exception:
        pass

# ---- LLM config from repo-root .env (one level above notebooks/) ----
load_dotenv(find_dotenv(usecwd=True))
API = os.getenv("OPENAI_API_KEY")
TIERS = {"weak": os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini"),
         "mid": os.getenv("LLM_MODEL_MID", "gpt-5.4-mini"),
         "strong": os.getenv("LLM_MODEL_STRONG", "gpt-5.4")}
if not API:
    raise SystemExit("No OPENAI_API_KEY found. Create a .env at the repo root and run locally.")

# ---- credit model ----
fp = DATA / "credit_german.csv"
if not fp.exists():
    pd.read_csv("https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv").to_csv(fp, index=False)
df = pd.read_csv(fp); df.columns = [c.strip().strip('"') for c in df.columns]
y = (df["credit_risk"].astype(int) == 1).astype(int)
X = df.drop(columns=["credit_risk", "personal_status_sex"])
NUM = X.select_dtypes("number").columns.tolist()
CAT = [c for c in X.columns if c not in NUM]
PROTECTED = ["sex", "gender", "foreign_worker", "foreign", "age", "marital"]
pre = ColumnTransformer([("num", StandardScaler(), NUM),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
model = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=3000))]).fit(X, y)
proba = model.predict_proba(X)[:, 1]
coef = model.named_steps["lr"].coef_[0]
names = model.named_steps["pre"].get_feature_names_out()
LO, HI = X[NUM].quantile(0.01), X[NUM].quantile(0.99)   # plausibility bounds
RECOURSE_NUM = [c for c in NUM if c not in PROTECTED]    # recourse may not use a protected feature

def orig_feature(expanded):
    if expanded.startswith("num__"):
        return expanded[5:]
    if expanded.startswith("cat__"):
        rest = expanded[5:]
        cands = [c for c in CAT if rest.startswith(c + "_")]
        return max(cands, key=len) if cands else rest
    return expanded
MAP = [orig_feature(n) for n in names]

def local_drivers(case_df, k=3):
    z = model.named_steps["pre"].transform(case_df).ravel()
    contrib = coef * z                       # signed contribution to approve-logit
    agg = {}
    for f, c in zip(MAP, contrib):
        agg[f] = agg.get(f, 0.0) + c
    return [f for f, _ in sorted(agg.items(), key=lambda kv: kv[1])[:k]]  # most negative

# ---- LLM ----
from openai import OpenAI
client = OpenAI()
SCHEMA = {"numeric_features": NUM, "categorical_features": CAT}
SYSTEM = ("You explain an automated CREDIT DECLINE to the applicant. You are given the "
    "applicant's record and the feature schema, but NOT the model internals. Infer the most "
    "likely reasons for the decline and propose ONE concrete, realistic change to a single "
    "NUMERIC feature that would most likely reverse the decision. Never cite protected "
    "attributes (sex, marital status, age, foreign-worker status) as reasons. Return STRICT "
    'JSON: {"reasons": [feature_name, ...up to 3], "recourse_feature": numeric_feature_name, '
    '"recourse_new_value": number}.')

def ask(model_name, case):
    msg = ("Feature schema: %s\nApplicant record: %s\nReturn only the JSON."
           % (json.dumps(SCHEMA), json.dumps(case)))
    r = client.chat.completions.create(model=model_name, temperature=0,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": msg}])
    return r.choices[0].message.content

def parse_json(txt):
    try:
        return json.loads(txt)
    except Exception:
        m = re.search(r"\{.*\}", txt, re.S)
        return json.loads(m.group(0)) if m else {}

def match(reasons, truth):
    low = " ".join(str(x).lower() for x in reasons)
    return float(np.mean([t.lower() in low for t in truth])) if truth else np.nan

# ---- evaluate over N moderately-declined applicants (near, but not at, the
#      boundary) where a single plausible recourse is informative, not trivial ----
band = np.where((proba >= 0.20) & (proba <= 0.45))[0]
order = band[np.argsort(np.abs(proba[band] - 0.35))][:N_CASES]
per_rows, summ_rows = [], []
for tier, mname in TIERS.items():
    fths, valids, prot = [], [], []
    for i in order:
        case = X.iloc[int(i)].to_dict()
        truth = local_drivers(X.iloc[[int(i)]])
        try:
            out = parse_json(ask(mname, case))
            f = match(out.get("reasons", []), truth)
            rf, rv = out.get("recourse_feature"), out.get("recourse_new_value")
            valid = 0.0
            # valid only if a NON-protected numeric feature is changed to a
            # PLAUSIBLE (within p1..p99) value that actually flips the decision
            if rf in RECOURSE_NUM and isinstance(rv, (int, float)) and LO[rf] <= rv <= HI[rf]:
                mod = X.iloc[[int(i)]].copy(); mod[rf] = rv
                valid = float(model.predict_proba(mod)[0, 1] >= 0.5)
            p = float(any(any(k in str(r).lower() for k in PROTECTED) for r in out.get("reasons", [])))
        except Exception:
            f, valid, p = np.nan, np.nan, np.nan
        fths.append(f); valids.append(valid); prot.append(p)
        per_rows.append({"tier": tier, "model": mname, "case": int(i),
                         "faithfulness": f, "recourse_valid": valid, "protected_mention": p})
    summ_rows.append({"tier": tier, "model": mname,
        "faithfulness_mean": np.nanmean(fths), "recourse_valid_rate": np.nanmean(valids),
        "protected_mention_rate": np.nanmean(prot), "n": int(np.sum(~np.isnan(fths)))})
per = pd.DataFrame(per_rows); summ = pd.DataFrame(summ_rows)
savetable(per, "llm_explanation_per_case")
savetable(summ, "llm_explanation_eval")

# ---- figure: faithfulness vs recourse validity by tier (grayscale) ----
plot = summ.melt(id_vars="tier", value_vars=["faithfulness_mean", "recourse_valid_rate"],
                 var_name="metric", value_name="score")
plot["metric"] = plot["metric"].map({"faithfulness_mean": "Faithfulness",
                                      "recourse_valid_rate": "Recourse validity"})
fig, ax = plt.subplots(figsize=(5.0, 3.4))
sns.barplot(data=plot, x="tier", y="score", hue="metric",
            palette=["0.30", "0.65"], edgecolor="black", ax=ax)
ax.set_ylim(0, 1); ax.set_xlabel(""); ax.set_ylabel("Score")
ax.legend(frameon=False, fontsize=9, title="")
savefig(fig, "fig_llm_faithfulness")

print("llm leg done | tiers:", list(TIERS.values()),
      "| faithfulness:", summ["faithfulness_mean"].round(3).tolist(),
      "| recourse valid:", summ["recourse_valid_rate"].round(3).tolist())


llm leg done | tiers: ['gpt-4o-mini', 'gpt-5.4-mini', 'gpt-5.4'] | faithfulness: [0.133, 0.333, 0.378] | recourse valid: [0.0, 0.067, 0.067]
